# v4 — Window-context Host Graph: GNN vs Transformer in a deployment-faithful setting
**Intelligent Adaptive IDS using Hybrid Transformer and Graph Neural Networks — NF-CSE-CIC-IDS2018-v3**

Kaggle: **GPU T4** → add NF-CSE-CIC-IDS2018-v3 → **Run All**.

### Why this notebook (findings of FINAL_V3, 10 seeds)
| Finding | Consequence here |
|---|---|
| Graph + sequences were built on a **sparse class-sample spread over 11 days** — host degrees and time gaps are artificial and cannot be reproduced live | **Context = ALL flows inside short time windows** (2 min + 1 min warm-up). The graph and the sequences are built exactly as a live system would build them. Sampling is applied only to the flows that carry the training loss. |
| GNN training **collapsed whole classes** (Bot / GoldenEye F1 = 0) in 3 of 10 seeds | **LayerNorm** after each GAT layer, lower GNN learning rate, train-fitted scaling of node features |
| Graph models had **~13% FAR on unseen hosts** vs 1% for the Transformer | FAR / DR reported for **all test flows** and separately for **flows from hosts never seen in training** |
| Two-stage was not significantly better than Transformer-only | Same 3 models, same windows, same seeds → fair re-test where the graph carries real information |
| Bootstrap CI bug for L3-merged rows | Fixed: the CI uses the same metric as the row |

### Pipeline
```
full file → count flows per (2-min window, class) → choose windows per class, spread over days
         → load ALL flows of the chosen windows (+1 min warm-up before each)
         → split by WINDOW (random-window / temporal) → train-only preprocessing
         → per-window host graph (Node = IP, Edge = flow) + per-host sequences inside the window
         → GATv2 | Transformer-only | Two-stage (GATv2 embeddings → Transformer)
```
Rules unchanged: no SMOTE, class-weighted loss, validation-only model selection, test used once per run, results saved after every run.

## 1. Environment

In [ ]:
import sys, subprocess, importlib.util
for pkg, mod in [("torch-geometric", "torch_geometric"), ("pyarrow", "pyarrow")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, gc, re, json, time, math, random, shutil, zipfile, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, ttest_rel, binomtest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
elif not os.environ.get("IDS_ALLOW_CPU"):
    raise RuntimeError("No GPU. Kaggle: Settings -> Accelerator -> GPU T4, then Run All.")

## 2. Configuration

In [ ]:
QUICK = False                     # True = 2 seeds, random-window split only (first check)

SPLIT_SEED = 42
SEEDS = [42, 43, 44, 45, 46]
SPLITS_TO_RUN = ["random_window", "temporal"]

# ---- windows (context) ----
WINDOW_MIN = 2                    # a window = 2 minutes of ALL traffic
MARGIN_SEC = 60                   # + 1 minute before it as warm-up context (no labels used from the warm-up)
N_WIN_PER_CLASS = 10              # windows chosen per attack class (spread over its days)
N_BENIGN_WIN = 40                 # attack-free windows (spread over all days)
MIN_CLASS_FLOWS_IN_WINDOW = 20    # prefer windows where the class really happens
MAX_WINDOW_FLOWS = 30_000         # memory guard (very dense flood windows are skipped when possible)
TRAIN_TARGET_CAP = 5_000          # flows per class that carry the TRAINING loss (context keeps everything)
CHUNK_FLOWS = 60_000              # windows are processed in chunks of at most this many flows
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
CHUNK_SIZE = 500_000

# ---- features (28: locked list without TTL / TCP-window fingerprints) ----
MODEL_FEATURES = [
    "IN_BYTES", "LONGEST_FLOW_PKT", "MAX_IP_PKT_LEN", "OUT_BYTES", "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT", "DURATION_IN", "FLOW_DURATION_MILLISECONDS", "CLIENT_TCP_FLAGS",
    "SRC_TO_DST_IAT_MAX", "SRC_TO_DST_IAT_AVG", "SRC_TO_DST_IAT_STDDEV", "TCP_FLAGS", "DURATION_OUT",
    "DST_TO_SRC_IAT_AVG", "DST_TO_SRC_IAT_MAX", "NUM_PKTS_UP_TO_128_BYTES", "OUT_PKTS",
    "DST_TO_SRC_IAT_STDDEV", "IN_PKTS", "L7_PROTO", "SERVER_TCP_FLAGS", "NUM_PKTS_512_TO_1024_BYTES",
    "DST_TO_SRC_SECOND_BYTES", "SRC_TO_DST_SECOND_BYTES", "SHORTEST_FLOW_PKT", "MIN_IP_PKT_LEN",
    "NUM_PKTS_256_TO_512_BYTES",
]
LOG1P_FEATURES = True
K = 32                            # previous flows of the same source host inside the window (+ current)

# ---- models ----
GNN_HIDDEN, GNN_HEADS, EMB_DIM, DROPOUT = 64, 2, 128, 0.20
D_MODEL, N_HEADS, N_LAYERS, FF_DIM = 128, 4, 2, 256

# ---- training ----
BATCH_SIZE = 512
LR = 1e-3                         # Transformer / Two-stage
GNN_LR = 5e-4                     # lower LR for the GNN (stability)
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 150
PATIENCE = 20
MIN_EPOCHS = 10

# ---- evaluation ----
TARGET_FAR = 0.01
MERGE_GROUPS = {"FTP-BruteForce": "FTP/SlowHTTP", "DoS_attacks-SlowHTTPTest": "FTP/SlowHTTP"}
N_BOOTSTRAP = 1000

if QUICK:
    SEEDS, SPLITS_TO_RUN = SEEDS[:2], ["random_window"]

_e = os.environ.get               # offline smoke-test overrides
MAX_EPOCHS = int(_e("IDS_MAX_EPOCHS", MAX_EPOCHS)); MIN_EPOCHS = min(MIN_EPOCHS, MAX_EPOCHS)
if _e("IDS_SEEDS"): SEEDS = [int(s) for s in _e("IDS_SEEDS").split(",")]
if _e("IDS_K"): K = int(_e("IDS_K"))
if _e("IDS_NWIN"): N_WIN_PER_CLASS = int(_e("IDS_NWIN")); N_BENIGN_WIN = 2 * N_WIN_PER_CLASS

WORK_DIR = Path("/kaggle/working/V4_WINDOWS") if Path("/kaggle/working").exists() else Path("./V4_WINDOWS")
RES, FIG, MOD, PRED = (WORK_DIR / d for d in ["results", "figures", "models", "predictions"])
for p in (RES, FIG, MOD, PRED):
    p.mkdir(parents=True, exist_ok=True)
WMS, MARGIN_MS = WINDOW_MIN * 60_000, MARGIN_SEC * 1000

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, bool))},
          open(RES / "run_config.json", "w"), indent=2)
print("Output:", WORK_DIR, "| splits:", SPLITS_TO_RUN, "| seeds:", SEEDS)

## 3. Locate NF-CSE-CIC-IDS2018-v3 (CSV or ZIP)

In [ ]:
SRC_IP, DST_IP, SRC_PORT, DST_PORT = "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT"
T_START, T_END, PROTO, LABEL, BIN_LABEL = "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "PROTOCOL", "Attack", "Label"
REQUIRED = [T_START, SRC_IP, DST_IP, LABEL]

def _has_schema(read_head):
    try:
        cols = [str(c).strip() for c in read_head().columns]
        return all(c in cols for c in REQUIRED)
    except Exception:
        return False

def find_source():
    roots = [Path(_e("IDS_DATA_DIR"))] if _e("IDS_DATA_DIR") else [Path("/kaggle/input")]
    for root in roots:
        for p in sorted(root.rglob("*.csv")):
            if "unsw" not in p.name.lower() and _has_schema(lambda: pd.read_csv(p, nrows=0)):
                return {"type": "csv", "path": p}
        for zp in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(zp) as z:
                    for m in z.namelist():
                        if m.lower().endswith(".csv") and "unsw" not in m.lower():
                            if _has_schema(lambda: pd.read_csv(z.open(m), nrows=0)):
                                return {"type": "zip", "path": zp, "member": m}
            except Exception:
                pass
    raise FileNotFoundError("NF-CSE-CIC-IDS2018-v3 CSV not found under /kaggle/input.")

def iter_chunks(src, usecols=None):
    kw = dict(chunksize=CHUNK_SIZE, low_memory=False, on_bad_lines="skip", usecols=usecols)
    if src["type"] == "csv":
        yield from pd.read_csv(src["path"], **kw)
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            yield from pd.read_csv(f, **kw)

SOURCE = find_source()
print("Dataset:", SOURCE)

def day_of(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors="coerce"), unit="ms", utc=True).dt.strftime("%Y-%m-%d")

## 4. Pass 1 — count flows per (2-minute window, class) over the whole file

In [ ]:
BAD = {"", "nan", "none", "attack"}
cnt = defaultdict(int)
t0 = time.time()
for ch in iter_chunks(SOURCE, usecols=[T_START, LABEL]):
    lab = ch[LABEL].astype(str).str.strip()
    ok = ~lab.str.lower().isin(BAD)
    b = pd.to_numeric(ch.loc[ok, T_START], errors="coerce") // WMS
    vc = pd.DataFrame({"b": b, "c": lab[ok]}).dropna().astype({"b": "int64"}).value_counts()
    for (bb, c), n in vc.items():
        cnt[(int(bb), c)] += int(n)
WIN = pd.Series(cnt).unstack(fill_value=0).sort_index()
WIN.index.name = "window"
win_total = WIN.sum(1)
win_attack = WIN.drop(columns=["Benign"], errors="ignore").sum(1)
win_day = pd.Series(pd.to_datetime(WIN.index.values * WMS, unit="ms", utc=True).strftime("%Y-%m-%d"), index=WIN.index)
class_day = WIN.groupby(win_day).sum().T
class_day.to_csv(RES / "full_dataset_class_by_day.csv")
print(f"Pass 1: {time.time() - t0:.0f}s | windows with traffic: {len(WIN):,} | flows: {int(win_total.sum()):,}")
print("Median flows per 2-min window:", int(win_total.median()), "| 99th percentile:", int(win_total.quantile(0.99)))

## 5. Choose windows
For every attack class: `N_WIN_PER_CLASS` windows where the class really happens, spread round-robin over its days.
Plus `N_BENIGN_WIN` attack-free windows spread over all days. A window is selected **once** (its "primary" class).

In [ ]:
rng_w = np.random.default_rng(SPLIT_SEED)
selected = {}

def spread_pick(cands, n):
    by_day = {d: list(rng_w.permutation(g.index.values)) for d, g in win_day.loc[cands].groupby(win_day.loc[cands])}
    out = []
    while len(out) < n and any(by_day.values()):
        for d in list(by_day):
            if by_day[d] and len(out) < n:
                out.append(int(by_day[d].pop()))
    return out

attack_classes = [c for c in WIN.columns if c != "Benign"]
for c in sorted(attack_classes, key=lambda c: WIN[c].sum()):          # rarest first
    free = WIN.index[(WIN[c] > 0) & ~WIN.index.isin(list(selected))]
    good = free[(WIN.loc[free, c] >= MIN_CLASS_FLOWS_IN_WINDOW) & (win_total.loc[free] <= MAX_WINDOW_FLOWS)]
    pick = spread_pick(good, N_WIN_PER_CLASS) if len(good) else []
    if len(pick) < N_WIN_PER_CLASS:                                    # top up: most class flows, smallest windows
        rest = free.difference(pick)
        rest = sorted(rest, key=lambda b: (win_total[b] > MAX_WINDOW_FLOWS, -WIN.loc[b, c]))
        pick += [int(b) for b in rest[: N_WIN_PER_CLASS - len(pick)]]
    for b in pick:
        selected[b] = c
if "Benign" in WIN.columns:
    free = WIN.index[(win_attack == 0) & (WIN["Benign"] > 0) & (win_total <= MAX_WINDOW_FLOWS) & ~WIN.index.isin(list(selected))]
    for b in spread_pick(free, N_BENIGN_WIN):
        selected[b] = "Benign"

win_info = pd.DataFrame({"primary": pd.Series(selected)})
win_info["day"] = win_day.loc[win_info.index].values
win_info["flows"] = win_total.loc[win_info.index].values
win_info["start"] = pd.to_datetime(win_info.index.values * WMS, unit="ms", utc=True)
win_info.to_csv(RES / "selected_windows.csv")
print(f"Selected windows: {len(win_info)} | flows inside them: {int(win_info.flows.sum()):,}")
display(win_info.groupby("primary").agg(windows=("flows", "size"), days=("day", "nunique"), flows=("flows", "sum")))

## 6. Pass 2 — load ALL flows of the selected windows (+ warm-up minute before each)

In [ ]:
USECOLS = list(dict.fromkeys([T_START, T_END, SRC_IP, DST_IP, SRC_PORT, DST_PORT, PROTO, LABEL] + MODEL_FEATURES))
SEL = set(win_info.index)
parts = []
t0 = time.time()
for ch in iter_chunks(SOURCE, usecols=lambda c: str(c).strip() in USECOLS):
    ch.columns = [str(c).strip() for c in ch.columns]
    t = pd.to_numeric(ch[T_START], errors="coerce")
    b = t // WMS
    main = b.isin(SEL)
    nb = b + 1
    ctx = nb.isin(SEL) & (t >= nb * WMS - MARGIN_MS)
    if main.any():
        parts.append(ch[main].assign(_win=b[main].astype("int64").values, _main=True))
    if ctx.any():
        parts.append(ch[ctx].assign(_win=nb[ctx].astype("int64").values, _main=False))
data = pd.concat(parts, ignore_index=True); del parts; gc.collect()
data[LABEL] = data[LABEL].astype(str).str.strip()
data = data[~data[LABEL].str.lower().isin(BAD)]
data["_ts"] = pd.to_datetime(pd.to_numeric(data[T_START], errors="coerce"), unit="ms", utc=True)
data = data.dropna(subset=["_ts", SRC_IP, DST_IP])
dkey = ["_win"] + [c for c in [T_START, T_END, SRC_IP, DST_IP, SRC_PORT, DST_PORT, PROTO, LABEL] if c in data.columns]
n_dup = int(data.duplicated(subset=dkey).sum())
data = data.drop_duplicates(subset=dkey)
data[SRC_IP] = data[SRC_IP].astype(str); data[DST_IP] = data[DST_IP].astype(str)
data = data.sort_values(["_win", "_ts"], kind="mergesort").reset_index(drop=True)
print(f"Pass 2: {time.time() - t0:.0f}s | rows {len(data):,} (warm-up rows {int((~data._main).sum()):,}) | duplicates removed {n_dup:,}")

CLASSES = sorted(data[LABEL].unique())
le = LabelEncoder().fit(CLASSES); N_CLASSES = len(CLASSES)
BENIGN_ID = int(le.transform(["Benign"])[0])
data["_y"] = le.transform(data[LABEL])
labelled = data[data._main]
display(labelled.groupby(LABEL).agg(flows=("_y", "size"), windows=("_win", "nunique")))
pd.DataFrame({"Item": ["Rows (main)", "Rows (warm-up)", "Windows", "Classes", "Unique src IPs", "Benign share (main rows)"],
              "Value": [int(data._main.sum()), int((~data._main).sum()), data._win.nunique(), N_CLASSES,
                        data[SRC_IP].nunique(), float((labelled._y == BENIGN_ID).mean())]}).to_csv(RES / "data_audit.csv", index=False)

def family(name):
    n = name.lower()
    if n == "benign": return "Benign"
    if "ddos" in n: return "DDoS"
    if "dos" in n: return "DoS"
    if any(s in n for s in ["xss", "sql", "web"]): return "Web"
    if "brute" in n: return "BruteForce"
    if "bot" in n: return "Bot"
    if "infil" in n: return "Infiltration"
    return "Other"
L2_NAMES = sorted({family(c) for c in CLASSES}); MAP_L2 = np.array([L2_NAMES.index(family(c)) for c in CLASSES])
L3M_NAMES = sorted({MERGE_GROUPS.get(c, c) for c in CLASSES}); MAP_L3M = np.array([L3M_NAMES.index(MERGE_GROUPS.get(c, c)) for c in CLASSES])

## 7. Split by WINDOW
* **random_window** — each class's windows are shuffled and divided 70/15/15. No flow, host graph or sequence crosses splits.
* **temporal** — each class's windows are ordered by time: earliest → train, later → val, latest → test.

Unseen-host evaluation is built in: every test metric is also reported on the test flows whose **source host never appears in any training window**.

In [ ]:
def split_windows(mode):
    rng = np.random.default_rng(SPLIT_SEED)
    out = {"train": [], "val": [], "test": []}
    for c, g in win_info.groupby("primary"):
        w = g.sort_values("start").index.values if mode == "temporal" else rng.permutation(g.index.values)
        n = len(w)
        n_te = max(1, int(round(n * TEST_FRAC))) if n >= 3 else 0
        n_va = max(1, int(round(n * VAL_FRAC))) if n >= 3 else 0
        n_tr = n - n_te - n_va
        out["train"] += list(w[:n_tr]); out["val"] += list(w[n_tr:n_tr + n_va]); out["test"] += list(w[n_tr + n_va:])
    return out

## 8. Train-only preprocessing, per-window host graphs, per-host sequences

In [ ]:
def to_num(part):
    X = part[MODEL_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return np.sign(X) * np.log1p(np.abs(X)) if LOG1P_FEATURES else X

def node_stats(src, dst):
    hosts, inv = np.unique(np.concatenate([src, dst]), return_inverse=True)
    s, d = inv[:len(src)], inv[len(src):]
    n = len(hosts)
    out_d, in_d = np.bincount(s, minlength=n), np.bincount(d, minlength=n)
    u_out = pd.Series(d).groupby(s).nunique().reindex(range(n), fill_value=0).values
    u_in = pd.Series(s).groupby(d).nunique().reindex(range(n), fill_value=0).values
    return np.log1p(np.stack([out_d, in_d, out_d + in_d, u_out, u_in], 1).astype(np.float32)), s, d

def build_windows_seq(part, k):
    """Row i = previous k-1 flows of the same source host INSIDE the same window, then flow i. -1 = padding."""
    n = len(part)
    t = part["_ts"].values.astype("datetime64[ns]").astype(np.int64) / 1e9
    g = pd.factorize(part["_win"].astype(str) + "|" + part[SRC_IP])[0]
    order = np.lexsort((np.arange(n), t, g))
    gs = g[order]
    pos = np.arange(n) - np.searchsorted(gs, gs, side="left")
    seq = np.full((n, k), -1, dtype=np.int64); r = np.arange(n)
    for j in range(k):
        lag = k - 1 - j; ok = pos >= lag
        seq[order[ok], j] = order[r[ok] - lag]
    valid = seq >= 0
    dt = np.zeros((n, k), dtype=np.float32)
    tgt = np.broadcast_to(t[:, None], (n, k))
    dt[valid] = np.log1p(np.maximum(tgt[valid] - t[seq[valid]], 0))
    return seq, dt

def prepare(wins):
    parts = {n: data[data._win.isin(w)].sort_values(["_win", "_ts"], kind="mergesort").reset_index(drop=True)
             for n, w in wins.items()}
    Xtr = to_num(parts["train"])
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    raw_nodes = {}
    for n, part in parts.items():
        raw_nodes[n] = []
        for w, idx in part.groupby("_win", sort=False).indices.items():
            x, s, d = node_stats(part[SRC_IP].values[idx], part[DST_IP].values[idx])
            raw_nodes[n].append((x, s, d, int(idx[0]), int(idx[-1]) + 1))
    nsc = StandardScaler().fit(np.concatenate([x for x, *_ in raw_nodes["train"]]))
    train_hosts = set(parts["train"][SRC_IP]) | set(parts["train"][DST_IP])
    rng = np.random.default_rng(SPLIT_SEED)
    S = {"imputer": imp, "scaler": sc, "node_scaler": nsc, "k": K}
    for n, part in parts.items():
        X = np.clip(sc.transform(imp.transform(to_num(part))), -10, 10).astype(np.float32)
        seq, dt = build_windows_seq(part, K)
        main = part["_main"].values
        if n == "train":            # loss on a capped, class-balanced subset of main flows
            tgt = np.zeros(len(part), bool)
            for c in range(N_CLASSES):
                ix = np.where(main & (part["_y"].values == c))[0]
                tgt[rng.choice(ix, min(len(ix), TRAIN_TARGET_CAP), replace=False) if len(ix) else ix] = True
        else:                       # evaluate on ALL main flows (natural mix inside the windows)
            tgt = main.copy()
        wins_t = [{"x": torch.tensor(nsc.transform(x).astype(np.float32), device=device),
                   "s": torch.tensor(s, dtype=torch.long, device=device), "d": torch.tensor(d, dtype=torch.long, device=device),
                   "r0": r0, "r1": r1} for x, s, d, r0, r1 in raw_nodes[n]]
        S[n] = {"df": part, "n": len(part), "X": torch.tensor(X, device=device),
                "y": torch.tensor(part["_y"].values, dtype=torch.long, device=device),
                "seq": torch.tensor(seq, device=device), "dt": torch.tensor(dt, device=device),
                "target": torch.tensor(np.where(tgt)[0], device=device), "wins": wins_t,
                "unseen_host": (~part[SRC_IP].isin(train_hosts)).values & main}
    return S

def chunks(P, shuffle=False, seed=0):
    order = np.random.default_rng(seed).permutation(len(P["wins"])) if shuffle else np.arange(len(P["wins"]))
    cur, size = [], 0
    for k_ in order:
        w = P["wins"][k_]; m = w["r1"] - w["r0"]
        if cur and size + m > CHUNK_FLOWS:
            yield cur; cur, size = [], 0
        cur.append(int(k_)); size += m
    if cur:
        yield cur

def merge(P, wins):
    xs, fs, fd, rows, off = [], [], [], [], 0
    for k_ in wins:
        w = P["wins"][k_]
        xs.append(w["x"]); fs.append(w["s"] + off); fd.append(w["d"] + off)
        rows.append(torch.arange(w["r0"], w["r1"], device=device)); off += w["x"].shape[0]
    s, d, r = torch.cat(fs), torch.cat(fd), torch.cat(rows)
    ea = P["X"][r]
    g = Data(x=torch.cat(xs), edge_index=torch.stack([torch.cat([s, d]), torch.cat([d, s])]),
             edge_attr=torch.cat([ea, ea]), flow_src=s, flow_dst=d, flow_attr=ea)
    return g, r

## 9. Models (LayerNorm-stabilised GATv2, Temporal Transformer)
| model | graph (who talks to whom, inside the window) | time (host's recent flows, inside the window) |
|---|---|---|
| `gatv2` | ✔ | ✘ |
| `transformer` | ✘ | ✔ raw flow features as tokens |
| `two_stage` | ✔ frozen GATv2 flow embeddings as tokens | ✔ |

In [ ]:
class GATEncoder(nn.Module):
    def __init__(self, node_in, edge_in):
        super().__init__()
        self.gat1 = GATv2Conv(node_in, GNN_HIDDEN, heads=GNN_HEADS, concat=True, edge_dim=edge_in, dropout=DROPOUT)
        self.ln1 = nn.LayerNorm(GNN_HIDDEN * GNN_HEADS)
        self.gat2 = GATv2Conv(GNN_HIDDEN * GNN_HEADS, GNN_HIDDEN, heads=1, concat=False, edge_dim=edge_in, dropout=DROPOUT)
        self.ln2 = nn.LayerNorm(GNN_HIDDEN)
        self.drop = nn.Dropout(DROPOUT)
        self.edge_proj = nn.Sequential(nn.Linear(2 * GNN_HIDDEN + edge_in, EMB_DIM), nn.LayerNorm(EMB_DIM), nn.ReLU(), nn.Dropout(DROPOUT))

    def node_embed(self, g):
        x = self.drop(F.elu(self.ln1(self.gat1(g.x, g.edge_index, g.edge_attr))))
        return F.elu(self.ln2(self.gat2(x, g.edge_index, g.edge_attr)))

    def flow_embed(self, z, g, loc):
        return self.edge_proj(torch.cat([z[g.flow_src[loc]], z[g.flow_dst[loc]], g.flow_attr[loc]], 1))


class GATv2Clf(nn.Module):
    def __init__(self, node_in, edge_in):
        super().__init__()
        self.gnn = GATEncoder(node_in, edge_in)
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(EMB_DIM, N_CLASSES))


class TemporalTransformer(nn.Module):
    def __init__(self, tok_in, k):
        super().__init__()
        self.inp = nn.Linear(tok_in + 1, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, k, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, FF_DIM, DROPOUT, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)

    def forward(self, tok, dt, pad):
        h = self.inp(torch.cat([tok, dt.unsqueeze(-1)], -1)) + self.pos[:, -tok.size(1):]
        return self.norm(self.enc(h, src_key_padding_mask=pad)[:, -1])


class SeqClf(nn.Module):
    def __init__(self, source, tok_in, k):
        super().__init__()
        self.source = source                      # "X" (Transformer-only) or "emb" (Two-stage)
        self.temporal = TemporalTransformer(tok_in, k)
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, N_CLASSES))

    def forward(self, P, idx):
        seq = P["seq"][idx]; pad = seq < 0
        tok = P[self.source][seq.clamp(min=0)].float().masked_fill(pad.unsqueeze(-1), 0.0)
        return self.head(self.temporal(tok, P["dt"][idx], pad))

## 10. Training (validation-only selection) and evaluation

In [ ]:
@torch.no_grad()
def gat_all(model, P):
    """Logits and flow embeddings for EVERY row of a split, window-chunk by window-chunk."""
    model.eval()
    L = torch.zeros(P["n"], N_CLASSES, device=device); E = torch.zeros(P["n"], EMB_DIM, device=device)
    for ws in chunks(P):
        g, r = merge(P, ws)
        loc = torch.arange(len(r), device=device)
        e = model.gnn.flow_embed(model.gnn.node_embed(g), g, loc)
        E[r] = e; L[r] = model.head(e)
    return L, E

@torch.no_grad()
def target_logits(model, P, bs=4096):
    if isinstance(model, GATv2Clf):
        return gat_all(model, P)[0][P["target"]]
    model.eval()
    t = P["target"]
    return torch.cat([model(P, t[b:b + bs]) for b in range(0, len(t), bs)])

def train_model(kind, S, seed):
    set_seed(seed)
    tr = S["train"]; F_in = tr["X"].shape[1]
    if kind == "gatv2":
        model, lr = GATv2Clf(tr["wins"][0]["x"].shape[1], F_in).to(device), GNN_LR
    else:
        model, lr = SeqClf("X" if kind == "transformer" else "emb", F_in if kind == "transformer" else EMB_DIM, S["k"]).to(device), LR
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8)
    ytr = tr["y"][tr["target"]].cpu().numpy()
    present = np.unique(ytr)
    w = np.ones(N_CLASSES, np.float32)
    w[present] = compute_class_weight("balanced", classes=present, y=ytr)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, device=device))
    is_tgt = torch.zeros(tr["n"], dtype=torch.bool, device=device); is_tgt[tr["target"]] = True
    yv = S["val"]["y"][S["val"]["target"]].cpu().numpy()
    best, state, best_ep, bad, hist = -1.0, None, 0, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); tot, cnt_ = 0.0, 0
        if kind == "gatv2":
            for ws in chunks(tr, shuffle=True, seed=seed * 1000 + ep):
                g, r = merge(tr, ws)
                loc = is_tgt[r].nonzero().squeeze(1)
                if loc.numel() == 0:
                    continue
                for b in range(0, loc.numel(), 8192):           # several steps per chunk, graph re-encoded each step
                    lb = loc[b:b + 8192]
                    z = model.gnn.node_embed(g)
                    loss = crit(model.head(model.gnn.flow_embed(z, g, lb)), tr["y"][r[lb]])
                    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                    tot += loss.item() * lb.numel(); cnt_ += lb.numel()
        else:
            t = tr["target"][torch.randperm(tr["target"].numel(), device=device)]
            for b in range(0, t.numel(), BATCH_SIZE):
                idx = t[b:b + BATCH_SIZE]
                loss = crit(model(tr, idx), tr["y"][idx])
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                tot += loss.item() * idx.numel(); cnt_ += idx.numel()
        vl = target_logits(model, S["val"])
        vf1 = f1_score(yv, vl.argmax(1).cpu().numpy(), average="macro", zero_division=0)
        sched.step(vf1)
        hist.append({"epoch": ep, "train_loss": tot / max(cnt_, 1), "val_macro_f1": vf1})
        if vf1 > best + 1e-5:
            best, best_ep, bad = vf1, ep, 0
            state = {k_: v.detach().clone() for k_, v in model.state_dict().items()}
        else:
            bad += 1
        if ep >= MIN_EPOCHS and bad >= PATIENCE:
            break
    model.load_state_dict(state)
    return model, {"best_epoch": best_ep, "stopped_epoch": ep, "best_val_macro_f1": best}, pd.DataFrame(hist)

def attack_threshold(val_logits, yv):
    sc = 1 - torch.softmax(val_logits, 1)[:, BENIGN_ID].cpu().numpy()
    b = sc[yv == BENIGN_ID]
    return float(np.quantile(b, 1 - TARGET_FAR)) if len(b) else 0.5

def evaluate(prob, yt, thr):
    p = prob.argmax(1); score = 1 - prob[:, BENIGN_ID]; att = yt != BENIGN_ID
    return {"L3 Macro-F1": f1_score(yt, p, labels=np.unique(yt), average="macro", zero_division=0),
            "L3-merged Macro-F1": f1_score(MAP_L3M[yt], MAP_L3M[p], labels=np.unique(MAP_L3M[yt]), average="macro", zero_division=0),
            "L2 Macro-F1": f1_score(MAP_L2[yt], MAP_L2[p], labels=np.unique(MAP_L2[yt]), average="macro", zero_division=0),
            "L3 MCC": matthews_corrcoef(yt, p),
            "L1 FAR @val-thr": float((score[~att] > thr).mean()) if (~att).any() else np.nan,
            "L1 DR @val-thr": float((score[att] > thr).mean()) if att.any() else np.nan,
            "L1 ROC-AUC": roc_auc_score(att, score) if 0 < att.mean() < 1 else np.nan,
            "n_flows": int(len(yt)), "benign_share": float((~att).mean())}, p, score

RESULTS, PERCLASS, SPLIT_INFO = [], [], []
safe = lambda t: re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

def run_and_log(kind, S, seed, split):
    t0 = time.time()
    model, info, hist = train_model(kind, S, seed)
    P_v, P_t = S["val"], S["test"]
    yv = P_v["y"][P_v["target"]].cpu().numpy(); yt = P_t["y"][P_t["target"]].cpu().numpy()
    thr = attack_threshold(target_logits(model, P_v), yv)
    prob = torch.softmax(target_logits(model, P_t), 1).cpu().numpy()
    unseen = P_t["unseen_host"][P_t["target"].cpu().numpy()]
    out = {}
    for subset, m_ in [("all test flows", np.ones(len(yt), bool)), ("unseen-host flows", unseen)]:
        if m_.sum() < 20:
            continue
        m, p, score = evaluate(prob[m_], yt[m_], thr)
        RESULTS.append({"split": split, "subset": subset, "model": kind, "seed": seed, **m, **info,
                        "seconds": round(time.time() - t0)})
        out[subset] = m
        if subset == "all test flows":
            pc = f1_score(yt, p, labels=np.arange(N_CLASSES), average=None, zero_division=0)
            PERCLASS.append({"split": split, "model": kind, "seed": seed, **dict(zip(CLASSES, pc))})
            np.savez_compressed(PRED / f"{split}__{kind}__seed{seed}.npz", y=yt, pred=p, score=score, unseen=unseen)
    hist.to_csv(RES / f"history__{split}__{kind}__seed{seed}.csv", index=False)
    pd.DataFrame(RESULTS).to_csv(RES / "all_runs.csv", index=False)
    pd.DataFrame(PERCLASS).to_csv(RES / "per_class_f1_all_runs.csv", index=False)
    a, u = out.get("all test flows", {}), out.get("unseen-host flows", {})
    print(f"  {split:13s} {kind:11s} seed={seed}  L3-F1={a.get('L3 Macro-F1', np.nan):.4f}  "
          f"FAR={a.get('L1 FAR @val-thr', np.nan):.4f}  DR={a.get('L1 DR @val-thr', np.nan):.4f}  | unseen hosts: "
          f"FAR={u.get('L1 FAR @val-thr', np.nan):.4f} DR={u.get('L1 DR @val-thr', np.nan):.4f}  "
          f"(best ep {info['best_epoch']}/{info['stopped_epoch']}, {round(time.time() - t0)}s)")
    return model

## 11. Main experiment

In [ ]:
for split in SPLITS_TO_RUN:
    print(f"\n==================== SPLIT: {split} ====================")
    wins = split_windows(split)
    S = prepare(wins)
    info = {"split": split, **{f"windows_{n}": len(w) for n, w in wins.items()},
            **{f"flows_{n}": S[n]["n"] for n in ["train", "val", "test"]},
            "train_loss_flows": int(S["train"]["target"].numel()),
            "test_benign_share": float((S["test"]["y"][S["test"]["target"]] == BENIGN_ID).float().mean()),
            "test_unseen_host_flows": int(S["test"]["unseen_host"].sum())}
    SPLIT_INFO.append(info); print(pd.DataFrame([info]).to_string(index=False))
    cls_cov = pd.DataFrame({n: S[n]["df"].loc[S[n]["df"]._main, LABEL].value_counts() for n in ["train", "val", "test"]}).fillna(0).astype(int)
    cls_cov.to_csv(RES / f"class_coverage__{split}.csv")
    missing = cls_cov.index[(cls_cov["test"] == 0) | (cls_cov["train"] == 0)].tolist()
    if missing:
        print("NOTE: classes missing from train or test in this split:", missing)
    for si, seed in enumerate(SEEDS):
        print(f"  --- seed {seed} ---")
        gat = run_and_log("gatv2", S, seed, split)
        for n in ["train", "val", "test"]:
            S[n]["emb"] = gat_all(gat, S[n])[1]
        if si == 0:
            torch.save(gat.state_dict(), MOD / f"gatv2__{split}__seed{seed}.pt")
        two = run_and_log("two_stage", S, seed, split)
        trf = run_and_log("transformer", S, seed, split)
        if si == 0:
            torch.save(two.state_dict(), MOD / f"two_stage__{split}__seed{seed}.pt")
            torch.save(trf.state_dict(), MOD / f"transformer__{split}__seed{seed}.pt")
        del gat, two, trf; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    del S; gc.collect()
pd.DataFrame(SPLIT_INFO).to_csv(RES / "split_overview.csv", index=False)

## 12. Summary — mean ± std over seeds, for all test flows and for unseen-host flows

In [ ]:
res = pd.DataFrame(RESULTS)
KEY = ["L3 Macro-F1", "L3-merged Macro-F1", "L2 Macro-F1", "L3 MCC", "L1 FAR @val-thr", "L1 DR @val-thr", "L1 ROC-AUC", "best_epoch"]
summary = res.groupby(["split", "subset", "model"], sort=False)[KEY].agg(["mean", "std"]).round(4)
summary.to_csv(RES / "summary_mean_std.csv"); display(summary)

## 13. Statistics (paired over seeds; bootstrap CI now uses the SAME metric as the row)

In [ ]:
def macro_f1_for(metric, y, p):
    if metric == "L3-merged Macro-F1":
        y, p = MAP_L3M[y], MAP_L3M[p]
    return f1_score(y, p, labels=np.unique(y), average="macro", zero_division=0)

def compare(split, a, b, metric):
    sub = res[(res.split == split) & (res.subset == "all test flows")]
    A = sub[sub.model == a].set_index("seed")[metric]; B = sub[sub.model == b].set_index("seed")[metric]
    seeds = sorted(set(A.index) & set(B.index))
    if len(seeds) < 2:
        return None
    A, B = A.loc[seeds].values, B.loc[seeds].values; d = A - B
    p_t = ttest_rel(A, B).pvalue
    p_w = wilcoxon(A, B).pvalue if np.any(d != 0) else np.nan
    za = np.load(PRED / f"{split}__{a}__seed{seeds[0]}.npz"); zb = np.load(PRED / f"{split}__{b}__seed{seeds[0]}.npz")
    y, pa, pb = za["y"], za["pred"], zb["pred"]
    rs = np.random.default_rng(0); boots = []
    for _ in range(N_BOOTSTRAP):
        i = rs.integers(0, len(y), len(y))
        boots.append(macro_f1_for(metric, y[i], pa[i]) - macro_f1_for(metric, y[i], pb[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    mc = []
    for s in seeds:
        y_, pa_ = (lambda z: (z["y"], z["pred"]))(np.load(PRED / f"{split}__{a}__seed{s}.npz"))
        pb_ = np.load(PRED / f"{split}__{b}__seed{s}.npz")["pred"]
        n01, n10 = int(((pa_ == y_) & (pb_ != y_)).sum()), int(((pa_ != y_) & (pb_ == y_)).sum())
        mc.append(binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0)
    verdict = "A better" if (d.mean() > 0 and p_t < 0.05 and lo > 0) else \
              "B better" if (d.mean() < 0 and p_t < 0.05 and hi < 0) else "no significant difference"
    return {"split": split, "metric": metric, "A": a, "B": b, "n_seeds": len(seeds), "mean diff (A-B)": d.mean(),
            "std diff": d.std(ddof=1), "A wins": f"{int((d > 0).sum())}/{len(d)}", "paired t p": p_t, "Wilcoxon p": p_w,
            "McNemar p<0.05": f"{sum(p < 0.05 for p in mc)}/{len(mc)}", "bootstrap 95% CI (1st seed)": f"[{lo:+.4f}, {hi:+.4f}]",
            "verdict": verdict}

rows = [compare(s, a, b, m) for s in SPLITS_TO_RUN
        for a, b in [("two_stage", "transformer"), ("two_stage", "gatv2"), ("transformer", "gatv2")]
        for m in ["L3 Macro-F1", "L3-merged Macro-F1"]]
stats = pd.DataFrame([r for r in rows if r]); stats.to_csv(RES / "statistical_tests.csv", index=False)
display(stats.round(4))

# FAR on unseen hosts: is the graph still raising false alarms?
far_u = res[res.subset == "unseen-host flows"].pivot_table(index=["split", "seed"], columns="model", values="L1 FAR @val-thr")
far_u.to_csv(RES / "far_unseen_hosts_per_seed.csv"); display(far_u.round(4))

## 14. Figures

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
COL = {"gatv2": "#8fa9bf", "transformer": "#c9a66b", "two_stage": "#2e5e8c"}
MODS = ["gatv2", "transformer", "two_stage"]
for subset in ["all test flows", "unseen-host flows"]:
    sub = res[res.subset == subset]
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
    for ax, met in zip(axes, ["L3 Macro-F1", "L3-merged Macro-F1", "L1 DR @val-thr", "L1 FAR @val-thr"]):
        x = np.arange(len(SPLITS_TO_RUN)); w = 0.8 / 3
        for j, m in enumerate(MODS):
            g = sub[sub.model == m].groupby("split")[met].agg(["mean", "std"]).reindex(SPLITS_TO_RUN)
            ax.bar(x + j * w - 0.4 + w / 2, g["mean"], w, yerr=g["std"].fillna(0), color=COL[m], label=m, capsize=3)
        ax.set_xticks(x); ax.set_xticklabels(SPLITS_TO_RUN); ax.set_title(met); ax.grid(axis="y", alpha=0.25)
        if "FAR" in met:
            ax.axhline(TARGET_FAR, ls="--", color="#999", lw=1)
    axes[0].legend(frameon=False)
    plt.suptitle(f"{subset} — mean ± std over {len(SEEDS)} seeds", y=1.03)
    plt.savefig(FIG / f"01_models__{safe(subset)}.png", dpi=160, bbox_inches="tight"); plt.show()

pcf = pd.DataFrame(PERCLASS)
for split in SPLITS_TO_RUN:
    Mx = pcf[pcf.split == split].groupby("model")[CLASSES].mean().reindex(MODS).T
    fig, ax = plt.subplots(figsize=(8, 0.42 * N_CLASSES + 1.5))
    ax.imshow(Mx.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels(MODS); ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES)
    for i in range(N_CLASSES):
        for j in range(3):
            ax.text(j, i, f"{Mx.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if Mx.values[i, j] > 0.6 else "black")
    ax.set_title(f"Per-class F1 — {split} (mean over seeds)")
    plt.savefig(FIG / f"02_per_class_f1__{split}.png", dpi=160, bbox_inches="tight"); plt.show()

## 15. Package

In [ ]:
zip_path = shutil.make_archive(str(WORK_DIR.parent / "V4_WINDOWS_RESULTS"), "zip", root_dir=str(WORK_DIR))
print("RUN COMPLETE ->", zip_path)

## How to read the results
1. **statistical_tests.csv** — in the window-context setting, does Two-stage (graph + time) beat Transformer-only?
2. **far_unseen_hosts_per_seed.csv** — are the graph models still unstable / producing many false alarms on new hosts?
3. **summary (unseen-host flows)** — detection on hosts never seen in training.
4. **class_coverage__*.csv** — classes with very few windows (e.g. SQL / XSS) give noisy per-class numbers.